# Airbnb Price Category Classification with PySpark

This notebook trains classification models to predict the price tier of an Airbnb listing. The target variable `price_category` has three classes: Low, Medium, and High, determined by the 33rd and 66th price percentiles. Both Logistic Regression and Random Forest classifiers are trained and evaluated.

## Imports

We bring in PySpark ML components for preprocessing and modeling, along with standard Python utilities for file path resolution.

In [1]:
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Start Spark Session

A Spark session is started with a descriptive app name. If one is already running in the environment, this will return the existing session.

In [2]:
spark = SparkSession.builder.appName("airbnb-classification-pyspark").getOrCreate()

26/05/02 01:19:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Settings

We define the list of candidate input file paths, the name of the target column, and the full set of features we want to use. Features are organized into categorical and numeric groups. Each feature maps to a list of possible column names to handle variation across different versions of the cleaned dataset.

In [3]:
LOCAL_PATHS = [
    "airbnb_cleaned",
    "airbnb_cleaned_single",
    "airbnb_cleaned.parquet",
    "airbnb_cleaned.csv",
    "listings_cleaned.csv",
    "listings.csv"
]

TARGET_COL = "price_category"

FEATURE_CANDIDATES = {
    "neighbourhood_group": ["neighbourhood_group", "neighbourhood_group_cleansed"],
    "neighbourhood": ["neighbourhood", "neighbourhood_cleansed"],
    "room_type": ["room_type"],
    "host_identity_verified": ["host_identity_verified"],
    "instant_bookable": ["instant_bookable"],
    "cancellation_policy": ["cancellation_policy"],
    "construction_year": ["construction_year"],
    "minimum_nights": ["minimum_nights"],
    "number_of_reviews": ["number_of_reviews"],
    "reviews_per_month": ["reviews_per_month"],
    "review_rate_number": ["review_rate_number", "review_scores_rating"],
    "calculated_host_listings_count": ["calculated_host_listings_count"],
    "availability_365": ["availability_365"],
}

CATEGORICAL_BASE = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type",
    "host_identity_verified",
    "instant_bookable",
    "cancellation_policy",
]

NUMERIC_BASE = [
    "construction_year",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "review_rate_number",
    "calculated_host_listings_count",
    "availability_365",
]

## Helper Functions

`load_data` iterates over candidate paths and loads the first one that exists, supporting both CSV and Parquet formats. `first_existing` picks the first available column name from a list of candidates, which is how we handle alternative column naming. `resolve_columns` applies this logic to all features and returns a mapping from canonical name to the actual column in the DataFrame. `safe_numeric_expr` and `safe_to_double` handle dirty numeric strings the same way as the cleaning notebook.

In [4]:
def load_data(spark):
    for local_path in LOCAL_PATHS:
        path = Path(local_path)
        if path.exists():
            print(f"Using local dataset: {local_path}")

            if local_path.endswith(".parquet"):
                return spark.read.parquet(local_path)

            return spark.read.csv(local_path, header=True, inferSchema=False)

    raise FileNotFoundError(f"No local dataset found. Checked: {LOCAL_PATHS}")


def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def resolve_columns(df):
    resolved = {}

    for canonical, candidates in FEATURE_CANDIDATES.items():
        chosen = first_existing(df, candidates)
        if chosen is not None:
            resolved[canonical] = chosen

    return resolved


def safe_numeric_expr(col_name):
    cleaned = F.regexp_replace(
        F.trim(F.col(col_name).cast("string")),
        r"[^0-9.-]",
        ""
    )

    return (
        F.when(
            (cleaned == "") |
            (cleaned == ".") |
            (cleaned == "-") |
            (cleaned == "-.") |
            (cleaned == ".-"),
            None
        )
        .when(cleaned.rlike(r"^-?\d+(\.\d+)?$"), cleaned.cast("double"))
        .otherwise(None)
    )


def safe_to_double(df, col_name, output_col=None):
    if output_col is None:
        output_col = col_name

    if col_name not in df.columns:
        return df

    return df.withColumn(output_col, safe_numeric_expr(col_name))

## Target Creation Function

`ensure_target` checks whether a valid `price_category` column already exists in the data. If not, it derives one from the price column by computing the 33rd and 66th percentiles and assigning Low, Medium, or High accordingly. Rows with null or zero prices are dropped before the quantiles are computed.

In [5]:
def ensure_target(df):
    if TARGET_COL in df.columns:
        valid_target_rows = df.where(
            F.col(TARGET_COL).isNotNull() &
            (F.trim(F.col(TARGET_COL).cast("string")) != "")
        ).count()

        print("Existing price_category rows:", valid_target_rows)

        if valid_target_rows > 0:
            print("Using existing target column:", TARGET_COL)
            return df

        print("Existing price_category is empty, so rebuilding it from price.")

    price_cols = [c for c in df.columns if "price" in c.lower()]
    print("Possible price columns:", price_cols)

    if "price" in df.columns:
        price_col = "price"
    elif "price_clean" in df.columns:
        price_col = "price_clean"
    elif price_cols:
        price_col = price_cols[0]
    else:
        raise ValueError("No price-related column found, so price_category cannot be created.")

    print("Deriving target from:", price_col)

    df = df.withColumn("price_num", safe_numeric_expr(price_col))

    print("Price check:")
    df.select(price_col, "price_num").show(20, truncate=False)

    df = df.filter(F.col("price_num").isNotNull())
    df = df.filter(F.col("price_num") > 0)

    row_count = df.count()
    print("Rows after price cleaning:", row_count)

    if row_count == 0:
        raise ValueError("No rows left after cleaning price. Check your price column.")

    quantiles = df.approxQuantile("price_num", [0.33, 0.66], 0.01)

    if len(quantiles) != 2:
        raise ValueError("Could not compute price quantiles.")

    q1, q2 = quantiles

    print("Low cutoff:", q1)
    print("High cutoff:", q2)

    df = df.withColumn(
        TARGET_COL,
        F.when(F.col("price_num") <= q1, F.lit("Low"))
         .when(F.col("price_num") <= q2, F.lit("Medium"))
         .otherwise(F.lit("High"))
    )

    print("New target distribution:")
    df.groupBy(TARGET_COL).count().orderBy(TARGET_COL).show(truncate=False)

    return df

## ML Pipeline Helper Functions

`build_preprocessor` constructs a PySpark ML Pipeline that encodes categorical columns with `StringIndexer` and `OneHotEncoder`, imputes numeric nulls with the median using `Imputer`, assembles all encoded features into a single vector with `VectorAssembler`, and finally label-encodes the target column. `evaluate_preds` computes accuracy, weighted precision, weighted recall, and F1 on a predictions DataFrame and prints a full breakdown.

In [6]:
def build_preprocessor(categorical_features, numeric_features):
    stages = []

    for c in categorical_features:
        stages.append(
            StringIndexer(
                inputCol=c,
                outputCol=f"{c}_idx",
                handleInvalid="keep"
            )
        )

    if categorical_features:
        stages.append(
            OneHotEncoder(
                inputCols=[f"{c}_idx" for c in categorical_features],
                outputCols=[f"{c}_oh" for c in categorical_features]
            )
        )

    if numeric_features:
        stages.append(
            Imputer(
                strategy="median",
                inputCols=numeric_features,
                outputCols=[f"{c}_imp" for c in numeric_features]
            )
        )

    assembler_inputs = [f"{c}_oh" for c in categorical_features] + [
        f"{c}_imp" for c in numeric_features
    ]

    if not assembler_inputs:
        raise ValueError("No usable feature columns found for modeling.")

    stages.append(
        VectorAssembler(
            inputCols=assembler_inputs,
            outputCol="features",
            handleInvalid="keep"
        )
    )

    stages.append(
        StringIndexer(
            inputCol=TARGET_COL,
            outputCol="label",
            handleInvalid="skip"
        )
    )

    return Pipeline(stages=stages)


def evaluate_preds(preds, title):
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    accuracy = evaluator_acc.evaluate(preds)
    precision = evaluator_precision.evaluate(preds)
    recall = evaluator_recall.evaluate(preds)
    f1 = evaluator_f1.evaluate(preds)

    print(f"\n{title}")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")

    print("Prediction breakdown:")
    preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show(50, truncate=False)

    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1_score": f1}

## Load Data

We call `load_data` to locate and read the cleaned dataset. The schema is printed so we can verify column types before preprocessing.

In [7]:
df = load_data(spark)

print("Original rows:", df.count())
print("Original columns:", len(df.columns))
df.printSchema()

Using local dataset: airbnb_cleaned
Original rows: 101661
Original columns: 28
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_identity_verified: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- country: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- instant_bookable: string (nullable = true)
 |-- cancellation_policy: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- construction_year: string (nullable = true)
 |-- price: string (nullable = true)
 |-- service_fee: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: string (nullable = true)
 |-- r

## Create or Validate Target

We call `ensure_target` to either confirm the existing `price_category` column is populated or rebuild it from the price column if needed.

In [8]:
df = ensure_target(df)

Existing price_category rows: 101661
Using existing target column: price_category


## Resolve Feature Columns

Using the candidate mappings defined in Settings, we resolve which actual column names are available in this particular dataset. Features are separated into categorical and numeric lists for the preprocessor.

In [9]:
resolved = resolve_columns(df)

feature_cols = [resolved[c] for c in FEATURE_CANDIDATES if c in resolved]
categorical_features = [resolved[c] for c in CATEGORICAL_BASE if c in resolved]
numeric_features = [resolved[c] for c in NUMERIC_BASE if c in resolved]

print("Resolved features:", feature_cols)
print("Categorical features:", categorical_features)
print("Numeric feature candidates:", numeric_features)

Resolved features: ['neighbourhood_group', 'neighbourhood', 'room_type', 'host_identity_verified', 'instant_bookable', 'cancellation_policy', 'construction_year', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']
Categorical features: ['neighbourhood_group', 'neighbourhood', 'room_type', 'host_identity_verified', 'instant_bookable', 'cancellation_policy']
Numeric feature candidates: ['construction_year', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']


## Clean Numeric Features

All numeric feature columns are cast to doubles using `safe_to_double`. We then verify that each column has at least one non-null value before including it in the model, discarding any that are entirely empty.

In [10]:
for col_name in numeric_features:
    df = safe_to_double(df, col_name)

usable_numeric = []

for col_name in numeric_features:
    if df.where(F.col(col_name).isNotNull()).limit(1).count() > 0:
        usable_numeric.append(col_name)

print("Usable numeric features:", usable_numeric)

Usable numeric features: ['construction_year', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']


## Filter Invalid Categorical Values

Categorical columns are restricted to known valid values. Boroughs are limited to the five New York City boroughs, room types to the four standard Airbnb types, and cancellation policies to the three standard tiers. Values outside these sets are replaced with null so the imputer handles them rather than passing garbage strings to the indexer.

In [11]:
valid_boroughs = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
valid_room_types = ["Entire home/apt", "Private room", "Shared room", "Hotel room"]
valid_cancellation = ["strict", "moderate", "flexible"]

if "neighbourhood_group" in df.columns:
    df = df.withColumn(
        "neighbourhood_group",
        F.when(F.col("neighbourhood_group").isin(valid_boroughs), F.col("neighbourhood_group")).otherwise(None)
    )

if "room_type" in df.columns:
    df = df.withColumn(
        "room_type",
        F.when(F.col("room_type").isin(valid_room_types), F.col("room_type")).otherwise(None)
    )

if "cancellation_policy" in df.columns:
    df = df.withColumn(
        "cancellation_policy",
        F.when(F.col("cancellation_policy").isin(valid_cancellation), F.col("cancellation_policy")).otherwise(None)
    )

if "instant_bookable" in df.columns:
    df = df.withColumn("instant_bookable", F.lower(F.col("instant_bookable").cast("string")))
    df = df.withColumn(
        "instant_bookable",
        F.when(F.col("instant_bookable").isin(["true", "false"]), F.col("instant_bookable")).otherwise(None)
    )

if "host_identity_verified" in df.columns:
    df = df.withColumn("host_identity_verified", F.lower(F.col("host_identity_verified").cast("string")))
    df = df.withColumn(
        "host_identity_verified",
        F.when(F.col("host_identity_verified").isin(["verified", "unconfirmed"]), F.col("host_identity_verified")).otherwise(None)
    )

## Create Modeling DataFrame

We select only the resolved feature columns and the target column, then drop rows where the target is null. The row count and class distribution are printed to confirm the data is ready for modeling.

In [12]:
model_cols = list(dict.fromkeys(feature_cols + [TARGET_COL]))
model_df = df.select(*model_cols).dropna(subset=[TARGET_COL])

row_count = model_df.count()

print("Shape:", (row_count, len(model_df.columns)))
print("Model target distribution:")
model_df.groupBy(TARGET_COL).count().orderBy(F.desc("count")).show(truncate=False)

if row_count == 0:
    raise ValueError("No rows available after target creation. Check that price_category has valid values.")

Shape: (101661, 14)
Model target distribution:
+--------------+-----+
|price_category|count|
+--------------+-----+
|High          |34230|
|Medium        |34164|
|Low           |33267|
+--------------+-----+



## Build and Apply Preprocessor

The preprocessing pipeline is fit on the full modeling DataFrame and then applied to produce a `(features, label)` DataFrame ready for training.

In [13]:
prep = build_preprocessor(categorical_features, usable_numeric)

prepared = prep.fit(model_df).transform(model_df).select("features", "label")

prepared.show(5, truncate=False)

26/05/02 01:19:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------------------+-----+
|features                                                                              |label|
+--------------------------------------------------------------------------------------+-----+
|(447,[247,440,441,442,443,444,445,446],[1.0,2012.0,22.0,1102022.0,4.0,3.0,362.0,89.0])|2.0  |
|(447,[337,440,441,442,443,444,445,446],[1.0,2012.0,3.0,7.0,5.0,3.0,179.0,89.0])       |2.0  |
|(447,[216,440,441,442,443,444,445,446],[1.0,2012.0,89.0,30.0,0.73,3.0,2.0,2.0])       |0.0  |
|(447,[250,440,441,442,443,444,445,446],[1.0,2012.0,32.0,712019.0,3.0,1.0,69.0,89.0])  |2.0  |
|(447,[244,440,441,442,443,444,445,446],[1.0,2012.0,27.0,562019.0,4.0,3.0,362.0,89.0]) |2.0  |
+--------------------------------------------------------------------------------------+-----+
only showing top 5 rows


## Train/Test Split

The prepared dataset is split 80/20 using a fixed random seed so results are reproducible.

In [14]:
train_df, test_df = prepared.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

Train rows: 81431


[Stage 83:>                                                         (0 + 3) / 3]

Test rows: 20230


## Logistic Regression

A multiclass Logistic Regression model is trained on the training set and evaluated on the test set. It acts as a linear baseline.

In [15]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)

lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)
lr_scores = evaluate_preds(lr_preds, "Logistic Regression Results")


Logistic Regression Results
Accuracy:  0.346
Precision: 0.345
Recall:    0.346
F1-score:  0.337
Prediction breakdown:


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|0.0  |0.0       |2835 |
|0.0  |1.0       |2741 |
|0.0  |2.0       |1237 |
|1.0  |0.0       |2630 |
|1.0  |1.0       |2863 |
|1.0  |2.0       |1271 |
|2.0  |0.0       |2626 |
|2.0  |1.0       |2724 |
|2.0  |2.0       |1303 |
+-----+----------+-----+



## Random Forest Classifier

A Random Forest with 100 trees is trained and evaluated. It can capture non-linear relationships and interactions between features that Logistic Regression cannot.

In [16]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)
rf_scores = evaluate_preds(rf_preds, "Random Forest Classifier Results")


Random Forest Classifier Results
Accuracy:  0.347
Precision: 0.367
Recall:    0.347
F1-score:  0.291
Prediction breakdown:


[Stage 183:>                                                        (0 + 3) / 3]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|0.0  |0.0       |4095 |
|0.0  |1.0       |2573 |
|0.0  |2.0       |145  |
|1.0  |0.0       |3895 |
|1.0  |1.0       |2725 |
|1.0  |2.0       |144  |
|2.0  |0.0       |3942 |
|2.0  |1.0       |2510 |
|2.0  |2.0       |201  |
+-----+----------+-----+



## Model Comparison

Accuracy, precision, recall, and F1 scores for both models are assembled into a summary table and printed side by side for easy comparison.

In [17]:
summary = spark.createDataFrame(
    [
        ("Logistic Regression", lr_scores["Accuracy"], lr_scores["Precision"], lr_scores["Recall"], lr_scores["F1_score"]),
        ("Random Forest Classifier", rf_scores["Accuracy"], rf_scores["Precision"], rf_scores["Recall"], rf_scores["F1_score"]),
    ],
    ["Model", "Accuracy", "Precision", "Recall", "F1_score"]
)

summary.show(truncate=False)

[Stage 186:>                                                        (0 + 1) / 1]

+------------------------+-------------------+------------------+-------------------+------------------+
|Model                   |Accuracy           |Precision         |Recall             |F1_score          |
+------------------------+-------------------+------------------+-------------------+------------------+
|Logistic Regression     |0.34607019278299556|0.3453892506511916|0.34607019278299556|0.3368807934640543|
|Random Forest Classifier|0.34705882352941175|0.3671733393934457|0.34705882352941175|0.2907021731219629|
+------------------------+-------------------+------------------+-------------------+------------------+



## Stop Spark

Uncomment and run this cell when you are done to release the Spark session.

In [18]:
# spark.stop()